In [ ]:
dbutils.widgets.text("DBT_SELECT", "vault gold", "dbt --select targets (space-separated)")
dbutils.widgets.text("CATALOG", "workspace", "Unity Catalog catalog")
dbutils.widgets.text("WAREHOUSE_ID", "53165753164ae80e", "SQL Warehouse ID")

In [ ]:
import subprocess, sys

print("Installing dbt-databricks...")
r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "dbt-databricks==1.9.3"],
    capture_output=True, text=True
)
if r.returncode != 0:
    raise RuntimeError(f"pip install failed:\n{r.stderr[-1000:]}")
print("dbt-databricks installed")

In [ ]:
import os, pathlib

dbt_select = dbutils.widgets.get("DBT_SELECT")
catalog     = dbutils.widgets.get("CATALOG")
warehouse_id = dbutils.widgets.get("WAREHOUSE_ID")

# Resolve host + token from Databricks runtime context
host  = spark.conf.get("spark.databricks.workspaceUrl")
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
http_path = f"/sql/1.0/warehouses/{warehouse_id}"

# Write a temporary profiles.yml
profiles_dir = pathlib.Path("/tmp/dbt_profiles")
profiles_dir.mkdir(exist_ok=True)
(profiles_dir / "profiles.yml").write_text(f"""cdc_gold:
  target: prod
  outputs:
    prod:
      type: databricks
      host: {host}
      http_path: {http_path}
      token: {token}
      catalog: {catalog}
      schema: gold
      threads: 4
""")
print(f"Profile written to {profiles_dir}/profiles.yml")
print(f"host={host}  http_path={http_path}")

In [ ]:
import subprocess, sys, pathlib

# Locate the dbt project directory (sibling of this notebook)
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
# notebook is at transformation/NB_run_dbt, project is at transformation/dbt_project
project_dir = pathlib.Path("/Workspace") / pathlib.Path(notebook_path).parent / "dbt_project"
print(f"dbt project dir: {project_dir}")

env = {**os.environ, "DBT_PROFILES_DIR": str(profiles_dir)}

def run_dbt(cmd):
    print(f"\n$ {' '.join(cmd)}")
    r = subprocess.run(cmd, capture_output=True, text=True, cwd=str(project_dir), env=env)
    if r.stdout: print(r.stdout[-3000:])
    if r.stderr: print(r.stderr[-1000:])
    if r.returncode != 0:
        raise RuntimeError(f"dbt command failed with exit code {r.returncode}")

run_dbt(["dbt", "deps"])

for target in dbt_select.split():
    run_dbt(["dbt", "build", "--select", target])

print("\ndbt build complete!")